<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 4 · Software Engineering, Reproducibility, and Deployment Basics
### Notebook 04 · Portfolio Capstone
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
The capstone is meant to look like a small professional project, not a scattered
collection of experiments.

## Setup
We begin by fixing the book root and import path so the capstone can load the
shared project code from a stable location.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "4_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Load the Capstone Dependencies
The first project cell imports the reusable data checks and the modeling tools.
That keeps the setup explicit before any data inspection begins.


In [ ]:
import importlib.util
from pathlib import Path

cwd = Path.cwd().resolve()
BOOK_ROOT = cwd
for candidate in (cwd, cwd.parent, cwd / "4_data"):
    if (candidate / "src").exists() or (candidate / "code").exists():
        BOOK_ROOT = candidate
        break

module_path = BOOK_ROOT / "src" / "data_checks.py"
spec = importlib.util.spec_from_file_location("data_checks", module_path)
assert spec is not None and spec.loader is not None
data_checks = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_checks)

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

load_project_health = data_checks.load_project_health
project_health_summary = data_checks.project_health_summary
validate_project_health = data_checks.validate_project_health


## Inspect and Validate the Project-Health Data
The capstone starts by loading the dataset, validating the expected columns, and
printing a compact preview. A portfolio project should make the raw input
visible before fitting a model.


In [ ]:
frame = load_project_health()
validate_project_health(frame)
print("Summary:", project_health_summary(frame))
print()
print(frame.head().to_string(index=False))


## Fit a Reproducible Baseline Pipeline
The next cell separates inputs from the target, defines the preprocessing steps,
fits a logistic-regression baseline, and prints a held-out evaluation report.


In [ ]:
feature_columns = [
    "domain",
    "owner_type",
    "active_weeks",
    "notebooks",
    "scripts",
    "test_files",
    "dashboard_views",
    "stale_days",
    "issue_count",
    "has_readme",
    "has_requirements",
]
target = "delivery_risk"

X = frame[feature_columns]
y = frame[target]

numeric = [
    "active_weeks",
    "notebooks",
    "scripts",
    "test_files",
    "dashboard_views",
    "stale_days",
    "issue_count",
]
categorical = ["domain", "owner_type", "has_readme", "has_requirements"]

# Scale numeric features and one-hot encode categoricals.
preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical,
        ),
    ]
)
model = Pipeline(
    [
        ("pre", preprocessor),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)

# Split off held-out data for the final score.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("accuracy =", round(accuracy_score(y_test, preds), 3))
print()
print(classification_report(y_test, preds, digits=3))


## Save a Short Capstone Summary
The notebook closes the coding workflow by writing a short Markdown summary into
`artifacts/`. That gives the capstone one more reusable deliverable outside the
notebook itself.


In [ ]:
report_path = BOOK_ROOT / "artifacts" / "capstone_summary.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_text = (
    "# Module 4 Capstone\n\n"
    "This project packages a small project-health dataset, reusable checks, "
    "a Streamlit dashboard, and a short evaluation notebook.\n"
)
report_path.write_text(report_text)
print(report_path.resolve())


### Report Summary
The final project should explain what problem it solves, how to run it, what the
dashboard adds, and what the next improvement would be.

### Where We Are Heading Next
The module ends here. The next real step is to publish the notebook, the code, the
dataset, and the README as one coherent portfolio repository.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">